# Caching is a property of the model

**Scenario:** a moderation service sends the same long policy document ahead of every report. The
policy never changes. Someone reads that prompt caching gives a large discount on a repeated prefix,
budgets for it, and ships.

The bill does not move.

Caching here is **a shop that may or may not keep your usual order behind the counter**. Some do,
some do not, and you find out by asking rather than by hoping.

## Mechanics

A repeated prefix can be reused by the provider so you are not charged full price for it twice. The
response tells you whether that happened.

| Field | Meaning |
|---|---|
| `usage.prompt_tokens` | Everything the model read this turn |
| `usage.prompt_tokens_details.cached_tokens` | How much of that was served from a cache |
| `usage.prompt_tokens_details.cache_write_tokens` | How much was put into a cache this turn |

Two things decide whether that number moves. The prefix has to be **byte stable**, because a cache
key is a hash and one changed character is a different key. And the upstream model has to offer
caching at all, which is not something your code can turn on.

Notice what is missing from that table. There is no field you set. You cannot request caching.

## The picture

![The same prefix, three models, three different answers](images/caching-by-model.svg)

The code is identical in all three lanes. Only the model id changes.

## The cost

```
saving = cached_tokens x prompt_rate x (1 - cache_discount)
```

Every term on the right comes from the response or the price list. If `cached_tokens` is zero, the
saving is zero no matter how stable your prefix is.

## The failure

A policy prefix long enough to be worth caching, and the same question asked three times.

In [1]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("13-cost-and-latency-at-volume/01-caching-is-a-property-of-the-model")

POLICY = ("You moderate a video platform. The rule table follows.\n" + "\n".join(
    f"RULE-{i:03d} class=C{i % 9} severity={1 + i % 5} appeal_days={7 + (i % 4) * 7} "
    f"reviewer=tier{1 + (i % 3)} region=R{i % 6}" for i in range(300)))

print(f"policy characters: {len(POLICY)}")

policy characters: 20679


The helper asks one question and reports what the provider said about caching. Nothing here requests
a cache, because there is nothing to request.

In [2]:
def ask(model, report_id):
    """One call. Returns tokens read, and how many of them were cached."""
    reply = client.chat.completions.create(
        model=model, max_tokens=40,
        messages=[{"role": "system", "content": POLICY},
                  {"role": "user", "content": f"Which reviewer tier handles RULE-{report_id}?"}])
    usage = reply.usage
    details = usage.prompt_tokens_details
    return usage.prompt_tokens, getattr(details, "cached_tokens", 0) or 0

Send the identical prefix three times to the model this repo defaults to, and watch the cached count.

In [3]:
default = model_for("default")
runs = [ask(default, rid) for rid in ("012", "145", "287")]

for i, (read, cached) in enumerate(runs, start=1):
    share = cached / read if read else 0
    print(f"  call {i}: read {read:6} tokens, cached {cached:6} ({share:.0%})")

assert runs[-1][1] > 0, f"{default} reported no cached tokens on a repeated prefix"

  call 1: read   8046 tokens, cached      0 (0%)
  call 2: read   8046 tokens, cached      0 (0%)
  call 3: read   8046 tokens, cached      0 (0%)


AssertionError: google/gemini-2.5-flash-lite reported no cached tokens on a repeated prefix

## The diagnosis

Zero cached tokens on all three calls, on a prefix that did not change by one character. The obvious
conclusion is that this model does not cache.

**That conclusion is wrong, and the next cell disproves it.** Keep it in mind while you read on,
because it is the most expensive mistake in this sub-module.

A cache has to be written before it can be read. The early calls in a series are the ones paying to
establish it, and a prefix does not become available to read back the instant it is sent. So a cold
spot check reports zero, and zero is what you would have taken to your capacity planning.

The second trap sits in the same measurement. Time these calls and the later ones come back faster.
That looks like evidence. It is not: a shorter round trip can come from routing, a warm connection
or load, and none of those reduce what you are charged. **Latency is not a receipt.** The only
evidence of a saving is `cached_tokens`.

## The fix

Warm the prefix first, then read the steady state. The first call establishes the cache and the
second is the one worth measuring.

In [4]:
def cache_profile(model):
    """Ask twice. The second call is the one that can be served from a cache."""
    ask(model, "012")
    read, cached = ask(model, "145")
    return {"model": model, "read": read, "cached": cached,
            "share": cached / read if read else 0.0}

Run it across the three models this repo has configured, all with the same prefix and the same code.

In [5]:
profiles = [cache_profile(model_for(role))
            for role in ("default", "reasoning", "small")]

for p in profiles:
    print(f"  {p['model']:28} read {p['read']:6}  cached {p['cached']:6}  {p['share']:6.1%}")

best = max(profiles, key=lambda p: p["share"])
print(f"\ncaches a repeated prefix: {best['model']} at {best['share']:.1%}")

  google/gemini-2.5-flash-lite read   8046  cached   7155   88.9%
  openai/gpt-5-nano            read   6628  cached   6528   98.5%
  mistralai/mistral-nemo       read   7750  cached      0    0.0%

caches a repeated prefix: openai/gpt-5-nano at 98.5%


Read that against the failure. The default model reported nothing while cold and a large share once
warm, so the earlier reading was about the measurement, not the model. One model in the set really
does report nothing, warm or cold, and that difference is only visible because both were measured
the same way.

Two lessons in one table. **Caching is a property of the model**, and **a cold measurement of it is
worthless.**

Now put a number on it, using the price the provider reports rather than a rate from a blog.

In [6]:
from vault import provider_truth

rates = provider_truth()["models"]


def saving_per_call(profile, discount=1.0):
    """Cash not spent on cached tokens. discount=1.0 assumes they are free."""
    rate = float(rates[profile["model"]]["prompt_usd_per_token"])
    return profile["cached"] * rate * discount


for p in profiles:
    print(f"  {p['model']:28} up to ${saving_per_call(p):.8f} per call")

  google/gemini-2.5-flash-lite up to $0.00071550 per call
  openai/gpt-5-nano            up to $0.00032640 per call
  mistralai/mistral-nemo       up to $0.00000000 per call


## The gate

The failure this prevents is subtle: someone appends a timestamp, a request id or a counter to the
front of the prompt, the key changes every call, and the saving quietly goes to zero. A test can pin
that without spending anything.

In [7]:
import hashlib


def prefix_key(messages):
    """What the provider hashes. Anything varying in here kills the cache."""
    head = messages[0]["content"]
    return hashlib.sha256(head.encode()).hexdigest()[:12]

Then the test itself, which needs no key and no network.

In [8]:
def test_the_prefix_is_stable_across_calls():
    keys = {prefix_key([{"role": "system", "content": POLICY},
                        {"role": "user", "content": q}])
            for q in ("Which reviewer tier handles RULE-012?",
                      "Which reviewer tier handles RULE-999?")}
    assert len(keys) == 1, f"prefix changed between calls: {keys}"


test_the_prefix_is_stable_across_calls()
print("gate holds: the cached prefix does not vary with the question")

gate holds: the cached prefix does not vary with the question


Put `f"Request at {time.time()}"` on the front of `POLICY` and this fails, which is exactly how the
saving disappears in production.

### Enterprise exploration

- A cold check said zero and a warm check said otherwise. What else in your monitoring is measured
  once, cold, and believed?
- Your prefix is stable today. What review stops someone adding a request id to it next quarter, and
  what does that cost per month once the saving quietly stops?
- A gateway routes to different upstream providers. What happens to your cache when it fails over,
  and what does that failure mode do to a budget built on the saving?
- Preferring a model because it caches is a trade off against quality and vendor lock in. At what
  request volume does that trade become worth making, and who signs it off?

### Key takeaways

- You cannot request caching. It is part of what a model id buys you.
- Measure warm. A cold reading reports zero and means nothing.
- `cached_tokens` is the only evidence. A faster reply is not a receipt.